# 1. Convert HTML to CSV

We copied the Abfall-ABC's HTML from the city of Würzburg: https://www.wuerzburg.de/themen/umwelt-klima/abfall-und-stadtreinigung/abfall-abc/index.html

In [ ]:
from bs4 import BeautifulSoup
import csv
import re

# Read the HTML file
with open("abfallabc.html", "r", encoding="utf-8") as file:
    soup = BeautifulSoup(file, "html.parser")

# Find all relevant divs
items = soup.find_all("div", class_="kt-info")

# Prepare data extraction
trash_data = []
for item in items:
    name = item.find("h1").text.strip()
    disposal = item.find("div", class_="kt-teaser").text.strip()
    # Strip prefix
    disposal = re.sub(
        r"^Entsorgungsmöglichkeit:\s*", "", disposal
    )  # Remove prefix and leading spaces
    trash_data.append([name, disposal])

# Write to CSV file
csv_filename = "trash_disposal.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Type of Trash", "Disposal"])
    writer.writerows(trash_data)

print(f"CSV file '{csv_filename}' has been created successfully!")


CSV file 'trash_disposal.csv' has been created successfully!


# 2. Group the entries in the CSV in a JSON object

In [3]:
import csv
import json
from collections import defaultdict

# Read the CSV file
csv_filename = "trash_disposal.csv"
categories = {
    "Wertstoffhof": [],
    "Restmülltonne": [],
    "Gelber Sack": [],
    "Glascontainer": [],
    "Biotonne": [],
    "Papiertonne": [],
    "Sonstige": [],
}

with open(csv_filename, "r", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # Skip header
    for row in reader:
        trash_type, disposal = row
        if "Wertstoff" in disposal:
            categories["Wertstoffhof"].append(trash_type)
        elif "Restmüll" in disposal:
            categories["Restmülltonne"].append(trash_type)
        elif "Gelber Sack" in disposal:
            categories["Gelber Sack"].append(trash_type)
        elif "Glascontainer" in disposal:
            categories["Glascontainer"].append(trash_type)
        elif "Biotonne" in disposal:
            categories["Biotonne"].append(trash_type)
        elif "Papiertonne" in disposal:
            categories["Papiertonne"].append(trash_type)
        else:
            categories["Sonstige"].append(trash_type)

# Convert to JSON
json_filename = "trash_categories.json"
with open(json_filename, "w", encoding="utf-8") as jsonfile:
    json.dump(categories, jsonfile, ensure_ascii=False, indent=4)

print(f"JSON file '{json_filename}' has been created successfully!")


JSON file 'trash_categories.json' has been created successfully!


# 3. Remove duplicates

Some items can fall into more than one category, which is not wanted for our game.

In [4]:
import json
import pandas as pd


def ensure_unique_elements(json_file):
    # Read the JSON file
    with open(json_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    # Ensure elements in each category are unique
    for key in data:
        if isinstance(data[key], list):  # Ensure it's a list before processing
            data[key] = list(set(data[key]))  # Remove duplicates

    # Remove items from 'Restmülltonne' if they exist in specified categories
    prioritized_categories = ["Gelber Sack", "Papiertonne", "Biotonne"]
    if "Restmülltonne" in data:
        restmuell_items = set(data["Restmülltonne"])
        for category in prioritized_categories:
            if category in data:
                restmuell_items -= set(data[category])
        data["Restmülltonne"] = list(restmuell_items)

    # Find duplicates among prioritized categories
    category_sets = {
        category: set(data.get(category, [])) for category in prioritized_categories
    }
    duplicates = set.intersection(*category_sets.values())

    if duplicates:
        print(
            "Duplicates found among Gelber Sack, Papiertonne, and Biotonne:", duplicates
        )
    else:
        print("No duplicates found among Gelber Sack, Papiertonne, and Biotonne.")

    # Compare Wertstoffhof to all other categories and remove duplicates from others
    wertstoffhof_items = set(data.get("Wertstoffhof", []))
    all_other_items = set()
    for category, items in data.items():
        if category != "Wertstoffhof" and isinstance(items, list):
            all_other_items.update(items)

    wertstoffhof_duplicates = wertstoffhof_items.intersection(all_other_items)

    if wertstoffhof_duplicates:
        print(
            "Duplicates found in Wertstoffhof compared to other categories:",
            wertstoffhof_duplicates,
        )
        # Remove duplicates from all other categories
        for category in data:
            if category != "Wertstoffhof" and isinstance(data[category], list):
                data[category] = list(set(data[category]) - wertstoffhof_duplicates)
    else:
        print("No duplicates found in Wertstoffhof compared to other categories.")

    # Compare Sonstige to all other categories and remove duplicates from others
    sonstige_items = set(data.get("Sonstige", []))
    all_other_items = set()
    for category, items in data.items():
        if category != "Sonstige" and isinstance(items, list):
            all_other_items.update(items)

    sonstige_duplicates = sonstige_items.intersection(all_other_items)

    if sonstige_duplicates:
        print(
            "Duplicates found in Sonstige compared to other categories:",
            sonstige_duplicates,
        )
        # Remove duplicates from all other categories
        for category in data:
            if category != "Sonstige" and isinstance(data[category], list):
                data[category] = list(set(data[category]) - sonstige_duplicates)
    else:
        print("No duplicates found in Sonstige compared to other categories.")

    # Write the cleaned data back to a new JSON file
    output_file = "cleaned_" + json_file
    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Processed JSON saved as {output_file}")

    # Final check: ensure no item is present in more than one category
    item_occurrences = {}
    for category, items in data.items():
        for item in items:
            item_occurrences.setdefault(item, []).append(category)

    duplicates_after_cleanup = {
        item: cats for item, cats in item_occurrences.items() if len(cats) > 1
    }

    if duplicates_after_cleanup:
        print("Warning: Some items are still present in multiple categories:")
        for item, categories in duplicates_after_cleanup.items():
            print(f"Item: {item}, Categories: {categories}")
    else:
        print("Final check passed: No item is present in more than one category.")


json_filename = "trash_categories.json"
ensure_unique_elements(json_filename)

No duplicates found among Gelber Sack, Papiertonne, and Biotonne.
Duplicates found in Wertstoffhof compared to other categories: {'Rasenschnitt', 'Baumschnitt', 'Autobatterie', 'Waschbecken', 'Autopflegemittel', 'Zeitungen, Zeitschriften', 'Dachziegel', 'Medikamente', 'Kataloge', 'Schaumstoffe', 'Teppiche', 'Pflanzenschutzmittel', 'Bohrmaschinen', 'Rollläden', 'Türen mit Glas', 'Sägen', 'Blech', 'Kloschüsseln', 'Öltanks', 'Keramik, -scherben', 'Kupfer', 'Sand', 'Packpapier', 'Kabel', 'Dosen', 'Leuchtstoffröhren', 'Plexiglas', 'Hausrat', 'Pappe', 'Batterien', 'Feuerlöscher', 'Bügeleisen', 'Föns', 'Spraydosen', 'Schläuche', 'Altöl (Motorenöl)', 'Tapetenreste', 'Papierverpackungen', 'Ceranglas, Ceranfelder', 'Putzmittelflaschen', 'Badkeramik', 'Öfen', 'Gusseisen', 'Wandfarben - lösemittelfrei (Blauer Engel)', 'Schrott', 'Öl', 'Solarkollektoren-Flüssigkeit', 'Reifen', 'Neonröhren', 'Betten', 'Eimer aus Kunststoff', 'Heckenschnitt', 'Thermoskannen', 'Steine, Fliesen', 'Eisenteile', 'Akkuboh

We clean the last two duplicates "Gläser" and "Kosmetika" manually :)